In [93]:
from CoolProp.CoolProp import PropsSI as PS
import numpy as np                            # arrays, for looping over temperatures
import matplotlib.pyplot as plt               # plots for your sensitivity figures

fl  = 'R1233zd(E)'
cpw = 4180.0
T_DC_out = 60 + 273.15 # Water temp. coming out of DC (K)
m_w = 28.7 # Water loop flow rate for DC cooling (kg/s) 
PP, SH, SC = 5.0, 4.0, 0 # Pinch (K), superheat (K), subcool (K)
eta_t, eta_p = 0.8, 0.75 # Efficiencies of ORC turbine and pump

# Pattern of fluid is PS(output, input1, value1, input2, value2, fluid)
# Q is vapour quality: 0 is saturated liquid, 1 is saturated vapour

def orc(T_evap, T_cond):
    P_e, P_c = PS('P','T',T_evap,'Q',1,fl), PS('P','T',T_cond,'Q',1,fl) # Gives evaporator and condenser pressures - check if this should be 0 for saturated liquid

    # 1: pump inlet (sat. liquid at T_cond, minus subcooling)
    T1 = T_cond - SC # Pump inlet temp.
    h1 = PS('H','T',T1,'Q',0,fl);  s1 = PS('S','T',T1,'Q',0,fl) # Enthalpy and entropy at pump inlet
    
    # 2: pump exit
    h2_ideal = PS('H','P',P_e,'S',s1,fl) # Calculates the enthalpy at pump exit if it were isentropic (pump just increases pressure from P_c to P_e and entropy stays the same)
    h2 = h1 + (h2_ideal - h1)/eta_p # Calculates the actual enthalpy at pump exit, accounting for pump efficiency (note, isentropic efficiency is used)
    
    # 3: WF liquid preheated to evaporation temperature
    h3 = PS('H','T',T_evap,'Q',0,fl)  # Enthalpy of saturated liquid at evaporator temperature
    
    # 4: turbine inlet
    h4 = PS('H','T',T_evap+SH,'P',P_e,fl);  s4 = PS('S','T',T_evap+SH,'P',P_e,fl) # Calculates the enthalpy and entropy at turbine inlet (superheated vapor)
    
    # 5: turbine outlet
    h5_ideal = PS('H','P',P_c,'S',s4,fl) # Calculates the enthalpy at turbine exit if it were isentropic (turbine just decreases pressure from P_e to P_c and entropy stays the same)
    h5   = h4 - eta_t*(h4 - h5_ideal) # Calculates the actual enthalpy at turbine exit, accounting for turbine efficiency (note, isentropic efficiency is used)

    # pinch at bubble point: water temp there = T_evap + PP
    Q_boil = m_w*cpw*(T_DC_out - (T_evap+PP)) # Heat for boiling + superheat (note the evaporator is counter-flow, so water first superheats + boils the working fluid, then preheats it)
    m_wf = Q_boil/(h4 - h3) # mass flow rate of working fluid in the ORC loop
    Q_in = m_wf*(h4 - h2) # Heat delivered to the ORC working fluid in the evaporator
    T_w_out_evap = T_DC_out - Q_in/(m_w*cpw) # check To >= T_evap-ish and cold-end pinch

    W_t, W_p = m_wf*(h4-h5), m_wf*(h2-h1)    # Work output from turbine and work input to pump
    Q_cond = m_wf*(h5 - h1) # Heat rejected to the condenser
    return dict(m_wf=m_wf, Q_in=Q_in, T_w_out_evap=T_w_out_evap, W_net=W_t-W_p, Q_cond=Q_cond, OCR_efficiency=100*(W_t-W_p)/Q_in)

Tcond is not chosen, it is set by the cold sink temperature. T_cond = wet bulb + approach + range + pinch.

Wet bulb - obtained from weather data

Approach temp - choose constant 4 degrees from "Modelling the influence paper"

Range - chosen, Claude and Thermo-economic comparison of air-cooled and cooling tower recommend around 10 F = 5K range (but then should iterate to find optimum)

Condensor pinch point - 5K, from the table 1 in Thermo-economic comparison of air-cooled and cooling tower 

In [108]:
T_wb = 2 + 273.15
T_PP_condensor = 5
T_approach = 4
T_range = 5
T_cond = T_wb + T_approach + T_range + T_PP_condensor

T_evap = 40 + 273.15 #Set T_evap

print(orc(T_evap, T_cond))

{'m_wf': 9.648495945458963, 'Q_in': 2074835.5884328468, 'T_w_out_evap': 315.8548031239447, 'W_net': 118429.42323982231, 'Q_cond': 1956406.1651930246, 'OCR_efficiency': 5.7078943459454425}


Now calculate the cooling tower pump power, based of the "Modelling the Influence of Climate on the Performance of the Organic Rankine Cycle for Industrial Waste Heat Recovery" paper.

This assumes a pressure drop aross the cooling tower pump of 100 kPa

In [109]:
# Call function to get Q_cond
Q_cond = orc(T_evap, T_cond)['Q_cond']

# Calculate required heat removal by the trim cooler
T_w_out_evap = orc(T_evap, T_cond)['T_w_out_evap']
T_DC_return = 40 + 273.15  # T of water flowing back into DC
Q_trim = m_w * cpw * (T_w_out_evap - T_DC_return)

# Add to get total heat rejected
Q_total_rejection = Q_cond + Q_trim

# Approximate density of water at condenser temperature
rho_w = 1000 # kg/m^3 (approximate density of water)

# Calculate volume flow rate of water in condensor loop (not mass flow rate)
V_w_ct = Q_total_rejection/(rho_w*cpw*(T_range))

# Define pressure drop in condenser loop (Pa)
dp = 100e3

# Define pump motor and impeller efficiencies
eta_m, eta_imp = 0.9, 0.78 

# Calculate the work required to pump the water through the condenser loop
W_ctp = V_w_ct * dp / (eta_m * eta_imp)
print(f"Nominal cooling tower pump power: {W_ctp/1e3:.2f} kW")

Nominal cooling tower pump power: 15.55 kW


A wet cooling tower transfers heat by evaporating a small fraction of the circulating water into an air stream. The energy balance requires that the heat rejected by your ORC condenser equals the enthalpy change of the air stream:$$\dot{Q}_{cond} = \dot{m}_{air} \cdot (h_{air,out} - h_{air,in})$$Inlet Air ($h_{air,in}$): Determined by ambient conditions (Dry-bulb temperature $T_{db}$, Wet-bulb temperature $T_{wb}$, and Relative Humidity $RH$).Outlet Air ($h_{air,out}$): Assumed to be saturated ($RH = 100\%$) at an exit temperature close to the warm water inlet temperature (typically estimated using a small approach temperature, e.g., $T_{air,out} = T_{water,in} - 2^\circ\text{C}$).Air Mass Flow Rate ($\dot{m}_{air}$): Calculated directly by rearranging the energy balance:$$\dot{m}_{air} = \frac{\dot{Q}_{cond}}{h_{air,out} - h_{air,in}}$$

In [110]:
from CoolProp.HumidAirProp import HAPropsSI

# --- Psychrometric Cooling Tower Air-Side Balance ---
# 1. Define ambient air conditions at the inlet - assume wat bulb temperature T_wb
RH_in = 0.50           # Assume 50% relative humidity for ambient air (check from where)
P_atm = 101325.0       # Atmospheric pressure (Pa)
T_db_in = 30 + 273.15 # Dry bulb temperature (replace with dry bulb temperature)

# Calculate inlet air specific enthalpy (J / kg of dry air)
h_air_in = HAPropsSI('H', 'T', T_wb, 'R', RH_in, 'P', P_atm)

# 2. Define exit air conditions
# Exit air is typically assumed saturated (100% RH) and warmed close to the incoming water temp
T_water_in_WT = T_wb + T_approach + T_range # Hot water T leaving condensor
T_air_out = T_water_in_WT - 2.0 # Exit air temperature approaches water-in temperature (find justification for this 2K)
RH_out = 1.0             # Saturated air leaving the wet deck - assume 100% saturated

# Calculate exit air specific enthalpy (J / kg of dry air)
h_air_out = HAPropsSI('H', 'T', T_air_out, 'R', RH_out, 'P', P_atm)

# 3. Solve for required air mass flow rate (m_air)
# Q_cond is in Watts
m_air = Q_cond / (h_air_out - h_air_in)
print(f"Calculated Air Mass Flow Rate: {m_air:.3f} kg/s")

# 4. Calculate volumetric air flow rate and fan power
from CoolProp.CoolProp import PropsSI as PS

# Calculates mass density of air directly (kg/m^3)
rho_air = PS('D', 'T', T_db_in, 'P', P_atm, 'HEOS::Air')
V_air = m_air / rho_air # Volume flow rate (m^3/s)

dp_air = 120.0          # Pressure drop across fill/eliminators (Pa) - find justification for this
eta_fan = 0.65          # Fan efficiency - justify
eta_motor_fan = 0.90    # Motor efficiency - justify

W_ct_fan = (V_air * dp_air) / (eta_fan * eta_motor_fan)
print(f"Cooling tower fan power: {W_ct_fan/1e3:.2f} kW")

Calculated Air Mass Flow Rate: 99.879 kg/s
Cooling tower fan power: 17.59 kW


I now calculate the Carnot efficiency of the same system. Carnot efficiency = $\eta_{max} = 1-\frac{T_C}{T_H}$.

I first take $T_C$ as the condensation temperature and $T_H$ as the evaporation temperature, giving the carnot efficiency of the vapour power cycle. This takes into account the fact of pinch points and non-infinite heat exchangers and can be compared to the ORC efficiency above.

Second, I also calculate a Carnot efficiency from the broad reservoir limit. This would be the Carnot efficiency if we assume infinite heat exchangers and therefore provides the ultimate upper cealing for efficiency. $T_C$ is the sink temperature which is the wet-bulb temperature, and $T_H$ is the temperature of the water from the DC (60 degrees)

In [119]:
# Calculating Carnot efficiency of the OCR loop
eta_Carnot_ORC = (1 - T_cond/T_evap)*100
print(f"Carnot efficiency of the ORC loop: {eta_Carnot_ORC:.2f}%")

# Calculating Carnot efficiency from the broad reservoir limit
eta_Carnot_broad = (1 - T_wb/T_DC_out)*100
print(f"Carnot efficiency from the broad reservoir limit: {eta_Carnot_broad:.2f}%")

Carnot efficiency of the ORC loop: 7.66%
Carnot efficiency from the broad reservoir limit: 17.41%
